In [2]:
import chromadb
import os

In [3]:
client = chromadb.PersistentClient(path="./chroma_db")

In [4]:
collection = client.get_or_create_collection(name="log_analytics")
print("Vector Database initialized in ./chroma_db")

Vector Database initialized in ./chroma_db


In [14]:
def load_logs_from_folder(folder_path):
    all_logs = []
    all_ids = []

    # Loop through your data folder
    for filename in os.listdir(folder_path):
        if filename.startswith('.'):
            continue
        file_path = os.path.join(folder_path, filename)
        
        if os.path.isfile(file_path):
            try:
                # We add 'errors="replace"' to swap bad characters with a '?' 
                # instead of crashing the whole script.
                with open(file_path, 'r') as f:
                    content = f.read()
                    all_logs.append(content)
                    all_ids.append(filename)
            except Exception as e:
                print(f"Skipping {filename} due to error: {e}")
                
    return all_logs, all_ids

In [22]:
logs, ids = load_logs_from_folder("/Users/adhithya/AI Powered Log Analytics/data")
print(f"Found {len(logs)} log files: {ids}")

Found 4 log files: ['deployment_meta.json', 'soa_server.xml', 'syslog_01.log', 'app_debug.txt']


In [24]:
collection.add(
documents=logs,
ids=ids
)
print("Ingestion Complete. Data is now searchable by AI.")

/Users/adhithya/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|███████████████████████████████████████████████████████| 79.3M/79.3M [03:06<00:00, 445kiB/s]


Ingestion Complete. Data is now searchable by AI.


In [25]:
count = collection.count()
print(f"Total items in DB: {count}")

sample = collection.get(ids=[ids[0]])
print("Sample Data in DB:", sample['documents'][0][:100], "...")

Total items in DB: 4
Sample Data in DB: {
"deployment_id": "dep_99x",
"environment": "production",
"orchestrator": "Kubernetes",
"ingress":  ...


In [27]:
query = "issues"

results = collection.query(
query_texts=[query],
n_results=2

)

print("USER QUERY:", query)
print("-" * 30)

for i in range(len(results['documents'][0])):
    print(f"MATCH {i+1} (Source: {results['ids'][0][i]}):")
    print(results['documents'][0][i][:200] + "...") # Show first 200 chars
    print("-" * 10)

USER QUERY: issues
------------------------------
MATCH 1 (Source: syslog_01.log):
Mar 19 02:15:01 oracle-prod-01 kernel: [12405.23] out of memory: kill process 8942 (python3) score 923 or sacrifice child
Mar 19 02:15:05 oracle-prod-01 systemd[1]: flask-app.service: Main process exi...
----------
MATCH 2 (Source: app_debug.txt):
[2026-03-19 10:05:22] INFO: Starting multithreaded search in /u01/app/oracle/shiphomes/
[2026-03-19 10:05:25] ERROR: Thread-4 raised Exception: Permission denied reading /u01/app/oracle/shiphomes/secr...
----------


In [28]:
%pip install fastapi

     |████████████████████████████████| 103 kB 4.3 MB/s eta 0:00:01
     |████████████████████████████████| 74 kB 11.7 MB/s eta 0:00:01
You should consider upgrading via the '/Users/adhithya/AI Powered Log Analytics/data/data_ingestion_notebook/ai_venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
